In [21]:

from pathlib import Path
import pandas as pd
import numpy as np

current = Path.cwd().resolve()

ROOT = next(
    (p for p in [current, *current.parents]
     if p.name == "Capstone_1"),
    None
)

if ROOT is None:
    raise FileNotFoundError("Capstone_1 folder not found.")

RAW_CSV = ROOT / "Part_2" / "data" / "raw" / "Tesla_Deaths.csv"
CLEAN_DIR = ROOT / "Part_2" / "data" / "clean"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

tesla_raw = pd.read_csv(RAW_CSV)

# Preserve the original DataFrame.
tesla_df = tesla_raw.copy()

# Remove leading/trailing whitespace from column names.
tesla_df.columns = tesla_df.columns.str.strip()

print("Raw shape:", tesla_raw.shape)
print("Working shape:", tesla_df.shape)

print("\nNormalized column names:")
for column in tesla_df.columns:
    print(repr(column))


Raw shape: (307, 24)
Working shape: (307, 24)

Normalized column names:
'Case #'
'Year'
'Date'
'Country'
'State'
'Description'
'Deaths'
'Tesla driver'
'Tesla occupant'
'Other vehicle'
'Cyclists/ Peds'
'TSLA+cycl / peds'
'Model'
'Autopilot claimed'
'Verified Tesla Autopilot Deaths'
'Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO'
'Unnamed: 16'
'Unnamed: 17'
'Source'
'Note'
'Deceased 1'
'Deceased 2'
'Deceased 3'
'Deceased 4'


In [22]:

important_columns = [
    "Case #",
    "Year",
    "Date",
    "Country",
    "State",
    "Description",
    "Deaths",
    "Tesla driver",
    "Tesla occupant",
    "Other vehicle",
    "Cyclists/ Peds",
    "TSLA+cycl / peds",
    "Model",
    "Autopilot claimed",
    "Verified Tesla Autopilot Deaths",
    "Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO"
]

available_columns = [
    column for column in important_columns
    if column in tesla_df.columns
]

display(tesla_df[available_columns].head(20))

print("\nData types:")
display(tesla_df[available_columns].dtypes.to_frame("Data Type"))


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,Cyclists/ Peds,TSLA+cycl / peds,Model,Autopilot claimed,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO
0,294.0,2022.0,1/17/2023,USA,CA,Tesla crashes into back of semi,1.0,1,-,-,-,1,-,-,-,-
1,293.0,2022.0,1/7/2023,Canada,-,Tesla crashes,1.0,1,-,-,-,1,-,-,-,-
2,292.0,2022.0,1/7/2023,USA,WA,"Tesla hits pole, catches on fire",1.0,-,1,-,-,1,-,-,-,-
3,291.0,2022.0,12/22/2022,USA,GA,Tesla crashes and burns,1.0,1,-,-,-,1,-,-,-,-
4,290.0,2022.0,12/19/2022,Canada,-,Tesla crashes into storefront,1.0,-,-,-,1,1,-,-,-,-
5,289.0,2022.0,12/18/2022,USA,CA,Tesla hit two other cars,1.0,-,1,-,-,-,-,-,-,-
6,288.0,2022.0,12/17/2022,USA,CA,Tesla hits pedestrian on highway,1.0,-,-,-,1,1,-,-,-,-
7,287.0,2022.0,12/11/2022,USA,CA,Tesla crashes into wall,1.0,1,-,-,-,-,-,-,-,-
8,286.0,2022.0,12/11/2022,USA,MO,Collision at intersection,1.0,-,-,1,-,-,-,-,-,-
9,285.0,2022.0,12/6/2022,Canada,-,"Tesla veers, collides with truck",1.0,1,-,-,-,-,-,-,-,-



Data types:


,Data Type
Case #,float64
Year,float64
Date,str
Country,str
State,str
Description,str
Deaths,float64
Tesla driver,str
Tesla occupant,str
Other vehicle,str


In [23]:

for column in ["Case #", "Year", "Deaths"]:
    if column not in tesla_df.columns:
        continue

    print("\n" + "=" * 60)
    print("COLUMN:", column)
    print("=" * 60)

    print("Data type:", tesla_df[column].dtype)
    print("Non-null values:", tesla_df[column].notna().sum())
    print("Unique values:", tesla_df[column].nunique(dropna=True))

    print("\nSorted values:")
    display(
        tesla_df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
        .sort_values(column, na_position="last")
    )



COLUMN: Case #
Data type: float64
Non-null values: 294
Unique values: 294

Sorted values:


,Case #,Count
294,1.0,1
293,2.0,1
292,3.0,1
291,4.0,1
290,5.0,1
...,...,...
4,291.0,1
3,292.0,1
2,293.0,1
1,294.0,1



COLUMN: Year
Data type: float64
Non-null values: 294
Unique values: 11

Sorted values:


,Year,Count
11,202.0,1
10,2013.0,2
9,2014.0,4
8,2015.0,5
5,2016.0,15
7,2017.0,11
4,2018.0,18
2,2019.0,46
3,2020.0,39
1,2021.0,58



COLUMN: Deaths
Data type: float64
Non-null values: 299
Unique values: 8

Sorted values:


,Deaths,Count
0,1.0,247
1,2.0,38
3,3.0,7
4,4.0,3
5,92.0,1
6,261.0,1
7,353.0,1
8,2013.0,1
2,NaN,8


In [24]:

# Convert copies of the columns to numeric for diagnosis only.
# The original working columns are not modified here.

year_numeric = pd.to_numeric(tesla_df["Year"], errors="coerce")
deaths_numeric = pd.to_numeric(tesla_df["Deaths"], errors="coerce")

suspicious_year = (
    year_numeric.notna()
    & ~year_numeric.between(1900, 2100)
)

suspicious_deaths = (
    deaths_numeric.notna()
    & (
        (deaths_numeric < 0)
        | (deaths_numeric > 100)
    )
)

print("Rows with suspicious year values:", suspicious_year.sum())
print("Rows with suspicious death counts:", suspicious_deaths.sum())

display(
    tesla_df.loc[
        suspicious_year | suspicious_deaths,
        available_columns
    ]
)


Rows with suspicious year values: 1
Rows with suspicious death counts: 3


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,Cyclists/ Peds,TSLA+cycl / peds,Model,Autopilot claimed,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO
162,132.0,202.0,9/17/2020,USA,GA,"Car loses control, hits bus shelter",1.0,NaN,NaN,NaN,1,1,3,1,NaN,1
295,NaN,NaN,NaN,NaN,NaN,NaN,261.0,89,41,101,26,149,NaN,30,16,24
296,NaN,NaN,NaN,NaN,NaN,NaN,353.0,117,48,130,46,210,NaN,38,19,27
301,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,2014,2015,2016,2017,2018,2019,2020,2021,NaN


In [25]:

print("Exact duplicate rows:", tesla_df.duplicated().sum())

duplicate_rows = tesla_df[
    tesla_df.duplicated(keep=False)
].sort_values("Case #", na_position="last")

print("\nRows involved in exact duplicates:")
display(duplicate_rows)

# Inspect rows where all core event fields are missing.
core_columns = [
    "Case #", "Year", "Date", "Country",
    "Description", "Deaths"
]

core_columns = [
    column for column in core_columns
    if column in tesla_df.columns
]

blank_core_rows = tesla_df[
    tesla_df[core_columns].isna().all(axis=1)
]

print("\nRows with all core event fields missing:", len(blank_core_rows))
display(blank_core_rows)


Exact duplicate rows: 4

Rows involved in exact duplicates:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Rows with all core event fields missing: 7


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:

unnamed_columns = [
    column for column in tesla_df.columns
    if column.lower().startswith("unnamed:")
]

print("Unnamed columns:", unnamed_columns)

for column in unnamed_columns:
    print("\nColumn:", column)
    print("Non-null values:", tesla_df[column].notna().sum())
    display(tesla_df[column].dropna().head(15))


Unnamed columns: ['Unnamed: 16', 'Unnamed: 17']

Column: Unnamed: 16
Non-null values: 292


0      https://web.archive.org/web/20221222203930/ht...
1      https://web.archive.org/web/20221222203930/ht...
2      https://web.archive.org/web/20221222203930/ht...
3      https://web.archive.org/web/20221222203930/ht...
4      https://web.archive.org/web/20221223203725/ht...
5      https://web.archive.org/web/20221213060612/ht...
6      https://web.archive.org/web/20221213060612/ht...
7      https://web.archive.org/web/20221213060612/ht...
8      https://www.ky3.com/2022/12/04/springfield-po...
9      https://web.archive.org/web/20221201065037/ht...
10     https://web.archive.org/web/20221201065037/ht...
11     https://web.archive.org/web/20221127191022/ht...
12     https://web.archive.org/web/20221201055652/ht...
13     https://web.archive.org/web/20221120105724/ht...
14     https://web.archive.org/web/20221112163911/ht...
Name: Unnamed: 16, dtype: str


Column: Unnamed: 17
Non-null values: 289


0      https://web.archive.org/web/20221222203930/ht...
1      https://web.archive.org/web/20221222203930/ht...
2      https://web.archive.org/web/20221222203930/ht...
3      https://web.archive.org/web/20221222203930/ht...
4      https://web.archive.org/web/20221223203725/ht...
5      https://web.archive.org/web/20221219025408/ht...
6      https://web.archive.org/web/20221220054727/ht...
7      https://web.archive.org/web/20221213060612/ht...
8      https://www.ky3.com/2022/12/04/springfield-po...
9      https://www.tvanouvelles.ca/2022/12/05/une-fe...
10     https://web.archive.org/web/20221201065037/ht...
11     https://web.archive.org/web/20221127191022/ht...
12     https://web.archive.org/web/20221201055652/ht...
13     https://web.archive.org/web/20221120105724/ht...
14     https://web.archive.org/web/20221112163911/ht...
Name: Unnamed: 17, dtype: str

In [27]:

# A candidate event record should contain a case number,
# a date, or a meaningful description.
# This is a diagnostic filter, not yet the final cleaned dataset.

event_mask = (
    tesla_df["Case #"].notna()
    | tesla_df["Date"].notna()
    | tesla_df["Description"].notna()
)

candidate_events = tesla_df.loc[event_mask].copy()
non_event_rows = tesla_df.loc[~event_mask].copy()

print("Total rows:", len(tesla_df))
print("Candidate event rows:", len(candidate_events))
print("Non-event rows:", len(non_event_rows))

print("\nCandidate event records:")
display(candidate_events.head())

print("\nNon-event rows:")
display(non_event_rows)


Total rows: 307
Candidate event rows: 295
Non-event rows: 12

Candidate event records:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
0,294.0,2022.0,1/17/2023,USA,CA,Tesla crashes into back of semi,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230118162813/ht...,NaN,NaN,NaN,NaN,NaN
1,293.0,2022.0,1/7/2023,Canada,-,Tesla crashes,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230109041434/ht...,NaN,Taren Singh Lal,NaN,NaN,NaN
2,292.0,2022.0,1/7/2023,USA,WA,"Tesla hits pole, catches on fire",1.0,-,1,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230107232745/ht...,NaN,NaN,NaN,NaN,NaN
3,291.0,2022.0,12/22/2022,USA,GA,Tesla crashes and burns,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,NaN,NaN,NaN,NaN,NaN
4,290.0,2022.0,12/19/2022,Canada,-,Tesla crashes into storefront,1.0,-,-,-,...,-,-,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...,NaN,NaN,NaN,NaN,NaN



Non-event rows:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
294,NaN,NaN,NaN,NaN,NaN,NaN,92.0,28,7,29,...,3,3,Abroad,NaN,Abroad,NaN,NaN,NaN,NaN,NaN
295,NaN,NaN,NaN,NaN,NaN,NaN,261.0,89,41,101,...,16,24,USA,NaN,USA,NaN,NaN,NaN,NaN,NaN
296,NaN,NaN,NaN,NaN,NaN,NaN,353.0,117,48,130,...,19,27,Total,NaN,Total,NaN,NaN,NaN,NaN,NaN
297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
300,NaN,NaN,NaN,NaN,NaN,NaN,3.0,6,5,16,...,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
301,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,2014,2015,2016,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:

print("Case number range:")
display(candidate_events["Case #"].describe())

print("\nYear distribution:")
display(
    candidate_events["Year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDeaths distribution:")
display(
    candidate_events["Deaths"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRows with missing case numbers:")
display(
    candidate_events[candidate_events["Case #"].isna()]
)

print("\nRows with missing dates:")
display(
    candidate_events[candidate_events["Date"].isna()]
)


Case number range:


count    294.000000
mean     147.500000
std       85.014705
min        1.000000
25%       74.250000
50%      147.500000
75%      220.750000
max      294.000000
Name: Case #, dtype: float64


Year distribution:


Year
202.0      1
2013.0     2
2014.0     4
2015.0     5
2016.0    15
2017.0    11
2018.0    18
2019.0    46
2020.0    39
2021.0    58
2022.0    95
NaN        1
Name: count, dtype: int64


Deaths distribution:


Deaths
1.0    247
2.0     38
3.0      6
4.0      3
NaN      1
Name: count, dtype: int64


Rows with missing case numbers:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
306,NaN,NaN,NaN,NaN,NaN,"Note: AutoPilot released Oct 14, 2015",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Rows with missing dates:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
306,NaN,NaN,NaN,NaN,NaN,"Note: AutoPilot released Oct 14, 2015",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:

print("Duplicate case numbers:")
duplicate_cases = candidate_events[
    candidate_events["Case #"].notna()
    & candidate_events["Case #"].duplicated(keep=False)
].sort_values("Case #")

display(duplicate_cases)

print("\nDuplicate date + description combinations:")
duplicate_event_keys = candidate_events[
    candidate_events.duplicated(
        subset=["Date", "Description"],
        keep=False
    )
].sort_values(["Date", "Description"])

display(duplicate_event_keys)


Duplicate case numbers:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4



Duplicate date + description combinations:


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4


In [30]:

for column in ["Unnamed: 16", "Unnamed: 17", "Source"]:
    if column in candidate_events.columns:
        print(f"\n{column}")
        print(
            candidate_events[column]
            .dropna()
            .astype(str)
            .head(10)
            .to_list()
        )



Unnamed: 16
[' https://web.archive.org/web/20221222203930/https://wfxl.com/news/local/man-dies-after-single-vehicle-accident-car-fire-in-lee-county-flowing-well-tesla-fatality-electric-car ', ' https://web.archive.org/web/20221222203930/https://wfxl.com/news/local/man-dies-after-single-vehicle-accident-car-fire-in-lee-county-flowing-well-tesla-fatality-electric-car ', ' https://web.archive.org/web/20221222203930/https://wfxl.com/news/local/man-dies-after-single-vehicle-accident-car-fire-in-lee-county-flowing-well-tesla-fatality-electric-car ', ' https://web.archive.org/web/20221222203930/https://wfxl.com/news/local/man-dies-after-single-vehicle-accident-car-fire-in-lee-county-flowing-well-tesla-fatality-electric-car ', ' https://web.archive.org/web/20221223203725/https://www.thesuburban.com/news/west_island_news/eyewitness-account-of-fatal-car-crash-in-lachine/article_a68b9e52-3ae4-55fb-8b2d-5cfcd1bcddd2.html ', ' https://web.archive.org/web/20221213060612/https://mynewsla.com/crime/2

In [31]:
# Select rows with a valid numeric case number.
case_numbers = pd.to_numeric(tesla_df["Case #"], errors="coerce")
event_mask = case_numbers.notna()

tesla_events = tesla_df.loc[event_mask].copy()
non_event_rows = tesla_df.loc[~event_mask].copy()

# Store the case number as an integer.
tesla_events["Case #"] = pd.to_numeric(
    tesla_events["Case #"], errors="coerce"
).astype("Int64")

print("Total source rows:", len(tesla_df))
print("Numbered accident records:", len(tesla_events))
print("Rows excluded from the event dataset:", len(non_event_rows))

print("\nCase number range:")
print(tesla_events["Case #"].min(), "to", tesla_events["Case #"].max())

print("\nMissing case numbers in selected event records:")
print(tesla_events["Case #"].isna().sum())

display(tesla_events.head())

Total source rows: 307
Numbered accident records: 294
Rows excluded from the event dataset: 13

Case number range:
1 to 294

Missing case numbers in selected event records:
0


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Verified Tesla Autopilot Deaths,Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO,Unnamed: 16,Unnamed: 17,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4
0,294,2022.0,1/17/2023,USA,CA,Tesla crashes into back of semi,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230118162813/ht...,NaN,NaN,NaN,NaN,NaN
1,293,2022.0,1/7/2023,Canada,-,Tesla crashes,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230109041434/ht...,NaN,Taren Singh Lal,NaN,NaN,NaN
2,292,2022.0,1/7/2023,USA,WA,"Tesla hits pole, catches on fire",1.0,-,1,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230107232745/ht...,NaN,NaN,NaN,NaN,NaN
3,291,2022.0,12/22/2022,USA,GA,Tesla crashes and burns,1.0,1,-,-,...,-,-,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,NaN,NaN,NaN,NaN,NaN
4,290,2022.0,12/19/2022,Canada,-,Tesla crashes into storefront,1.0,-,-,-,...,-,-,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...,NaN,NaN,NaN,NaN,NaN


In [32]:
# Parse dates while retaining the original date strings for reference.
tesla_events["Date_original"] = tesla_events["Date"]

tesla_events["Date"] = pd.to_datetime(
    tesla_events["Date"],
    errors="coerce"
)

print("Missing or invalid dates:", tesla_events["Date"].isna().sum())

print("\nDate range:")
print("Earliest:", tesla_events["Date"].min())
print("Latest:  ", tesla_events["Date"].max())

print("\nRecords with missing or invalid dates:")
display(
    tesla_events.loc[
        tesla_events["Date"].isna(),
        ["Case #", "Year", "Date_original", "Country", "State", "Description"]
    ]
)

Missing or invalid dates: 0

Date range:
Earliest: 2013-04-02 00:00:00
Latest:   2023-01-17 00:00:00

Records with missing or invalid dates:


,Case #,Year,Date_original,Country,State,Description


In [33]:
tesla_events["Year_numeric"] = pd.to_numeric(
    tesla_events["Year"], errors="coerce"
)

tesla_events["Deaths_numeric"] = pd.to_numeric(
    tesla_events["Deaths"], errors="coerce"
)

# Compare the recorded year with the year obtained from the parsed date.
tesla_events["Date_year"] = tesla_events["Date"].dt.year

year_issues = tesla_events.loc[
    tesla_events["Year_numeric"].isna()
    | (tesla_events["Year_numeric"] != tesla_events["Date_year"])
]

death_issues = tesla_events.loc[
    tesla_events["Deaths_numeric"].isna()
    | (tesla_events["Deaths_numeric"] <= 0)
    | (tesla_events["Deaths_numeric"] > 20)
]

print("Records where Year is missing or disagrees with the date:")
display(
    year_issues[
        ["Case #", "Year", "Date", "Date_year", "Description"]
    ]
)

print("\nRecords with missing or unusual death counts:")
display(
    death_issues[
        ["Case #", "Date", "Description", "Deaths"]
    ]
)

Records where Year is missing or disagrees with the date:


,Case #,Year,Date,Date_year,Description
0,294,2022.0,2023-01-17,2023,Tesla crashes into back of semi
1,293,2022.0,2023-01-07,2023,Tesla crashes
2,292,2022.0,2023-01-07,2023,"Tesla hits pole, catches on fire"
26,268,2020.0,2022-09-16,2022,Tesla loses control and crashes into bus shel...
162,132,202.0,2020-09-17,2020,"Car loses control, hits bus shelter"



Records with missing or unusual death counts:


,Case #,Date,Description,Deaths


In [34]:
# Rename unnamed columns while preserving their contents.
rename_map = {
    "Unnamed: 16": "Source_URL_1",
    "Unnamed: 17": "Source_URL_2"
}

tesla_events = tesla_events.rename(columns=rename_map)

print("Updated columns:")
print(tesla_events.columns.tolist())

print("\nExample source links:")
source_columns = [
    col for col in ["Source_URL_1", "Source_URL_2", "Source"]
    if col in tesla_events.columns
]

display(tesla_events[source_columns].head(10))

Updated columns:
['Case #', 'Year', 'Date', 'Country', 'State', 'Description', 'Deaths', 'Tesla driver', 'Tesla occupant', 'Other vehicle', 'Cyclists/ Peds', 'TSLA+cycl / peds', 'Model', 'Autopilot claimed', 'Verified Tesla Autopilot Deaths', 'Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO', 'Source_URL_1', 'Source_URL_2', 'Source', 'Note', 'Deceased 1', 'Deceased 2', 'Deceased 3', 'Deceased 4', 'Date_original', 'Year_numeric', 'Deaths_numeric', 'Date_year']

Example source links:


,Source_URL_1,Source_URL_2,Source
0,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230118162813/ht...
1,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230109041434/ht...
2,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20230107232745/ht...
3,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...,https://web.archive.org/web/20221222203930/ht...
4,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...,https://web.archive.org/web/20221223203725/ht...
5,https://web.archive.org/web/20221213060612/ht...,https://web.archive.org/web/20221219025408/ht...,https://web.archive.org/web/20221219025408/ht...
6,https://web.archive.org/web/20221213060612/ht...,https://web.archive.org/web/20221220054727/ht...,https://web.archive.org/web/20221220054727/ht...
7,https://web.archive.org/web/20221213060612/ht...,https://web.archive.org/web/20221213060612/ht...,https://web.archive.org/web/20221213060612/ht...
8,https://www.ky3.com/2022/12/04/springfield-po...,https://www.ky3.com/2022/12/04/springfield-po...,https://www.ky3.com/2022/12/04/springfield-po...
9,https://web.archive.org/web/20221201065037/ht...,https://www.tvanouvelles.ca/2022/12/05/une-fe...,https://www.tvanouvelles.ca/2022/12/05/une-fe...


In [35]:
print("Final event dataset dimensions:", tesla_events.shape)

print("\nDuplicate case numbers:", tesla_events["Case #"].duplicated().sum())

print("\nMissing values in key fields:")
key_columns = [
    col for col in ["Case #", "Date", "Year", "Country", "Description", "Deaths"]
    if col in tesla_events.columns
]

display(tesla_events[key_columns].isna().sum().to_frame("Missing values"))

print("\nData types:")
display(tesla_events[key_columns].dtypes.to_frame("Data type"))

display(tesla_events.head())

Final event dataset dimensions: (294, 28)

Duplicate case numbers: 0

Missing values in key fields:


,Missing values
Case #,0
Date,0
Year,0
Country,0
Description,0
Deaths,0



Data types:


,Data type
Case #,Int64
Date,datetime64[us]
Year,float64
Country,str
Description,str
Deaths,float64


,Case #,Year,Date,Country,State,Description,Deaths,Tesla driver,Tesla occupant,Other vehicle,...,Source,Note,Deceased 1,Deceased 2,Deceased 3,Deceased 4,Date_original,Year_numeric,Deaths_numeric,Date_year
0,294,2022.0,2023-01-17,USA,CA,Tesla crashes into back of semi,1.0,1,-,-,...,https://web.archive.org/web/20230118162813/ht...,NaN,NaN,NaN,NaN,NaN,1/17/2023,2022.0,1.0,2023
1,293,2022.0,2023-01-07,Canada,-,Tesla crashes,1.0,1,-,-,...,https://web.archive.org/web/20230109041434/ht...,NaN,Taren Singh Lal,NaN,NaN,NaN,1/7/2023,2022.0,1.0,2023
2,292,2022.0,2023-01-07,USA,WA,"Tesla hits pole, catches on fire",1.0,-,1,-,...,https://web.archive.org/web/20230107232745/ht...,NaN,NaN,NaN,NaN,NaN,1/7/2023,2022.0,1.0,2023
3,291,2022.0,2022-12-22,USA,GA,Tesla crashes and burns,1.0,1,-,-,...,https://web.archive.org/web/20221222203930/ht...,NaN,NaN,NaN,NaN,NaN,12/22/2022,2022.0,1.0,2022
4,290,2022.0,2022-12-19,Canada,-,Tesla crashes into storefront,1.0,-,-,-,...,https://web.archive.org/web/20221223203725/ht...,NaN,NaN,NaN,NaN,NaN,12/19/2022,2022.0,1.0,2022


In [36]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

output_path = CLEAN_DIR / "tesla_deaths_preliminary_clean.csv"

tesla_events.to_csv(output_path, index=False)

print("Preliminary cleaned dataset saved to:")
print(output_path)

Preliminary cleaned dataset saved to:
C:\Material\Repo\Projects\Capstone_1\Part_2\data\clean\tesla_deaths_preliminary_clean.csv


In [37]:
# Work on a fresh copy of the preliminary event dataset.
tesla_clean = tesla_events.copy()

# Preserve the original year field.
tesla_clean["Year_original"] = tesla_clean["Year"]

# Derive calendar year from the parsed event date.
tesla_clean["Event_year"] = tesla_clean["Date"].dt.year.astype("Int64")

# Keep a numeric representation of the source year for comparisons.
tesla_clean["Year_source_numeric"] = pd.to_numeric(
    tesla_clean["Year_original"], errors="coerce"
)

# Flag missing or inconsistent source years.
tesla_clean["Year_review_flag"] = (
    tesla_clean["Year_source_numeric"].isna()
    | (tesla_clean["Year_source_numeric"] != tesla_clean["Event_year"])
)

# Flag implausible source-year values for manual review.
tesla_clean["Year_anomaly_flag"] = (
    tesla_clean["Year_source_numeric"].notna()
    & (
        (tesla_clean["Year_source_numeric"] < 1900)
        | (tesla_clean["Year_source_numeric"] > 2100)
    )
)

print("Records flagged for year review:",
      tesla_clean["Year_review_flag"].sum())

print("Records flagged for anomalous year values:",
      tesla_clean["Year_anomaly_flag"].sum())

display(
    tesla_clean.loc[
        tesla_clean["Year_review_flag"],
        [
            "Case #", "Year_original", "Date",
            "Event_year", "Year_review_flag", "Year_anomaly_flag"
        ]
    ]
)

Records flagged for year review: 5
Records flagged for anomalous year values: 1


,Case #,Year_original,Date,Event_year,Year_review_flag,Year_anomaly_flag
0,294,2022.0,2023-01-17,2023,True,False
1,293,2022.0,2023-01-07,2023,True,False
2,292,2022.0,2023-01-07,2023,True,False
26,268,2020.0,2022-09-16,2022,True,False
162,132,202.0,2020-09-17,2020,True,True


In [38]:
final_clean_path = CLEAN_DIR / "tesla_deaths_clean.csv"

tesla_clean.to_csv(final_clean_path, index=False)

print("Cleaned dataset saved to:")
print(final_clean_path)

print("\nDataset dimensions:", tesla_clean.shape)
print("Unique case numbers:", tesla_clean["Case #"].nunique())
print("Duplicate case numbers:", tesla_clean["Case #"].duplicated().sum())
print("Missing parsed dates:", tesla_clean["Date"].isna().sum())

Cleaned dataset saved to:
C:\Material\Repo\Projects\Capstone_1\Part_2\data\clean\tesla_deaths_clean.csv

Dataset dimensions: (294, 33)
Unique case numbers: 294
Duplicate case numbers: 0
Missing parsed dates: 0


In [39]:

from pathlib import Path
import pandas as pd
import numpy as np

# Load the cleaned dataset produced by the previous steps.
ROOT = Path.cwd()

while ROOT.name != "Capstone_1" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

CLEAN_CSV = ROOT / "Part_2" / "data" / "clean" / "tesla_deaths_clean.csv"

tesla_clean = pd.read_csv(CLEAN_CSV)

# Make a separate analysis copy; preserve tesla_clean unchanged.
analysis_df = tesla_clean.copy()

# Normalize whitespace in column names.
analysis_df.columns = analysis_df.columns.str.strip()

# Convert dash markers and blank strings to missing values
# in the analysis copy only.
analysis_df = analysis_df.replace(
    to_replace=r"^\s*-\s*$",
    value=np.nan,
    regex=True
)

analysis_df = analysis_df.replace(r"^\s*$", np.nan, regex=True)

# Ensure the event date and event year are parsed consistently.
analysis_df["Date"] = pd.to_datetime(
    analysis_df["Date"],
    errors="coerce"
)

analysis_df["Event_year"] = analysis_df["Date"].dt.year

# Create numeric analysis columns from the recorded count fields.
count_fields = [
    "Deaths",
    "Tesla driver",
    "Tesla occupant",
    "Other vehicle",
    "Cyclists/ Peds",
    "TSLA+cycl / peds",
    "Verified Tesla Autopilot Deaths",
    "Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO"
]

for col in count_fields:
    if col in analysis_df.columns:
        analysis_df[f"{col}_count"] = pd.to_numeric(
            analysis_df[col],
            errors="coerce"
        )

# Summarize missingness and confirm dimensions.
print("Analysis dataset shape:", analysis_df.shape)
print("\nMissing values in key fields:")
display(
    analysis_df[
        ["Case #", "Date", "Event_year", "Country", "Description", "Deaths"]
    ].isna().sum().to_frame("Missing values")
)

print("\nNumeric count-field summary:")
numeric_count_cols = [
    f"{col}_count" for col in count_fields
    if f"{col}_count" in analysis_df.columns
]
display(analysis_df[numeric_count_cols].describe().T)

# Save a distinct analysis-ready copy.
ANALYSIS_DIR = ROOT / "Part_2" / "data" / "clean"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_CSV = ANALYSIS_DIR / "tesla_deaths_analysis_ready.csv"
analysis_df.to_csv(ANALYSIS_CSV, index=False)

print("\nAnalysis-ready dataset saved to:", ANALYSIS_CSV)


Analysis dataset shape: (294, 41)

Missing values in key fields:


,Missing values
Case #,0
Date,0
Event_year,0
Country,0
Description,0
Deaths,0



Numeric count-field summary:


,count,mean,std,min,25%,50%,75%,max
Deaths_count,294.0,1.200680,0.513171,1.0,1.0,1.0,1.0,4.0
Tesla driver_count,117.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
Tesla occupant_count,43.0,1.116279,0.390927,1.0,1.0,1.0,1.0,3.0
Other vehicle_count,110.0,1.181818,0.510115,1.0,1.0,1.0,1.0,4.0
Cyclists/ Peds_count,44.0,1.045455,0.210707,1.0,1.0,1.0,1.0,2.0
TSLA+cycl / peds_count,181.0,1.160221,0.449404,1.0,1.0,1.0,1.0,4.0
Verified Tesla Autopilot Deaths_count,16.0,1.187500,0.403113,1.0,1.0,1.0,1.0,2.0
Verified Tesla Autopilot Deaths + All Deaths Reported to NHTSA SGO_count,24.0,1.125000,0.337832,1.0,1.0,1.0,1.0,2.0



Analysis-ready dataset saved to: c:\Material\Repo\Projects\Capstone_1\Part_2\data\clean\tesla_deaths_analysis_ready.csv


In [40]:

# Confirm that the new analysis copy retains the expected event records.
print("Rows:", len(analysis_df))
print("Unique case numbers:", analysis_df["Case #"].nunique())
print("Duplicate case numbers:", analysis_df["Case #"].duplicated().sum())

print("\nEvent-year distribution:")
display(
    analysis_df["Event_year"]
    .value_counts()
    .sort_index()
    .rename_axis("Event year")
    .to_frame("Recorded incidents")
)

print("\nDash markers remaining in selected fields:")
for col in [
    "Tesla driver",
    "Tesla occupant",
    "Other vehicle",
    "Cyclists/ Peds",
    "Model",
    "Autopilot claimed"
]:
    if col in analysis_df.columns:
        print(f"{col}: {(analysis_df[col].astype('string').str.strip() == '-').sum()}")


Rows: 294
Unique case numbers: 294
Duplicate case numbers: 0

Event-year distribution:


,Recorded incidents
Event year,
2013,2
2014,4
2015,5
2016,15
2017,11
2018,18
2019,46
2020,39
2021,58



Dash markers remaining in selected fields:
Tesla driver: 0
Tesla occupant: 0
Other vehicle: 0
Cyclists/ Peds: 0
Model: 0
Autopilot claimed: 0
